# Data Preprocessing - Olympic Countries Efficiency

This notebook handles data cleaning and preprocessing steps:
- Data validation
- Missing value handling
- Outlier detection and treatment
- Data normalization
- Saving processed data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.data.data_loader import (
    load_data, validate_data, handle_missing_values, 
    remove_outliers, get_data_info
)

sns.set_style('whitegrid')
print('Libraries loaded successfully!')

## Load and Validate Data

In [ ]:
# Load data
data_path = '../data/raw/olympic_countries_efficiency.csv'
df = load_data(data_path)

print(f"Initial dataset shape: {df.shape}")

# Validate
try:
    validate_data(df)
    print("Data validation passed!")
except ValueError as e:
    print(f"Validation error: {e}")

## Handle Missing Values

In [ ]:
# Display missing values before
print("Missing values before handling:")
print(df.isnull().sum())

# Handle missing values
df_clean = handle_missing_values(df, strategy='median')

print("\nMissing values after handling:")
print(df_clean.isnull().sum())

## Handle Duplicates

In [ ]:
# Remove duplicates
print(f"Duplicates before removal: {df_clean.duplicated().sum()}")

df_clean = df_clean.drop_duplicates()

print(f"Duplicates after removal: {df_clean.duplicated().sum()}")
print(f"Dataset shape after removing duplicates: {df_clean.shape}")

## Detect and Handle Outliers

In [ ]:
# Identify outliers using IQR method
print("Detecting outliers using IQR method...\n")

outlier_cols = ['population', 'gdp_per_capita', 'total_medals', 'athletes_sent']

for col in outlier_cols:
    if col in df_clean.columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
        print(f"{col:.<30} {outliers} outliers")

## Remove Outliers

## Feature Type Corrections

In [ ]:
# Ensure correct data types
print("Current data types:")
print(df_clean.dtypes)

# Convert Year to integer
if 'Year' in df_clean.columns:
    df_clean['Year'] = df_clean['Year'].astype('int32')

# Ensure medal columns are numeric
medal_cols = ['Gold', 'Silver', 'Bronze', 'total_medals']
for col in medal_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print("\nData types after correction:")
print(df_clean.dtypes)

## Summary Statistics After Cleaning

In [ ]:
print("\nSummary Statistics After Cleaning:")
df_clean.describe().T

## Data Quality Report

In [ ]:
print("="*60)
print("DATA QUALITY REPORT")
print("="*60)

print(f"\nOriginal dataset: {df.shape}")
print(f"Cleaned dataset: {df_clean.shape}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")

print(f"\nMissing values: {df_clean.isnull().sum().sum()}")
print(f"Duplicates: {df_clean.duplicated().sum()}")
print(f"Data completeness: {100 - (df_clean.isnull().sum().sum() / (df_clean.shape[0] * df_clean.shape[1]) * 100):.1f}%")

print("\n" + "="*60)

## Save Processed Data

In [ ]:
# Create processed data directory if it doesn't exist
Path('../data/processed').mkdir(parents=True, exist_ok=True)

# Save processed data
output_path = '../data/processed/processed_data.csv'
df_clean.to_csv(output_path, index=False)

print(f"Processed data saved to: {output_path}")
print(f"Final dataset shape: {df_clean.shape}")